# 08 — Anomaly Explainability & Investigation Context

This notebook turns anomaly scores into **human-readable investigation context**.

It does **not** retrain any anomaly detector.

Goals:
- load the final saved scores from Isolation Forest, One-Class SVM and Autoencoder
- align anomaly scores with the original test transactions
- normalize scores into comparable percentiles using **validation scores as reference**
- build simple risk tiers and model-agreement indicators
- explain suspicious transactions using training-only contextual statistics
- inspect Autoencoder reconstruction error at feature level
- export structured investigation packets that can later be passed to an LLM

Important principle:

> The anomaly detectors detect and score.  
> The explanation layer contextualizes why a transaction looks unusual.  
> The LLM will later summarize and investigate; it will not decide whether a transaction is anomalous.


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from scipy import sparse


## Paths

In [2]:
RAW_PATH = Path("../data/raw/bs140513_032310.csv")
PROCESSED_PATH = Path("../data/processed")
MODELS_PATH = Path("../models")
RESULTS_PATH = Path("../results")
SCORES_PATH = RESULTS_PATH / "scores"
INVESTIGATION_PATH = RESULTS_PATH / "investigations"

INVESTIGATION_PATH.mkdir(parents=True, exist_ok=True)


## Load raw BankSim transactions

In [3]:
df = pd.read_csv(RAW_PATH)

# BankSim string columns often contain apostrophes in the raw file.
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = df[col].str.strip("'")

print("Full dataset:", df.shape)
df.head()


C:\Users\bench\AppData\Local\Temp\ipykernel_21828\483948335.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


Full dataset: (594643, 10)


,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,C1093826151,4,M,28007,M348934600,28007,es_transportation,4.55,0
1,0,C352968107,2,M,28007,M348934600,28007,es_transportation,39.68,0
2,0,C2054744914,4,F,28007,M1823072687,28007,es_transportation,26.89,0
3,0,C1760612790,3,M,28007,M348934600,28007,es_transportation,17.25,0
4,0,C757503768,5,M,28007,M348934600,28007,es_transportation,35.72,0


## Recreate the temporal split

In [4]:
train_raw = df[df["step"] <= 119].copy()
valid_raw = df[
    (df["step"] >= 120)
    & (df["step"] <= 149)
].copy()
test_raw = df[df["step"] >= 150].copy()

print("Train:", train_raw.shape)
print("Valid:", valid_raw.shape)
print("Test :", test_raw.shape)


Train: (374914, 10)
Valid: (108423, 10)
Test : (111306, 10)


Expected row counts:

- Train: **374,914**
- Validation: **108,423**
- Test: **111,306**

The row order must match the matrices and score arrays produced by notebook 02 and the model notebooks.


## Load final anomaly scores

In [5]:
iforest_valid_scores = np.load(
    SCORES_PATH / "iforest_valid_scores.npy"
)
iforest_test_scores = np.load(
    SCORES_PATH / "iforest_test_scores.npy"
)

ocsvm_valid_scores = np.load(
    SCORES_PATH / "ocsvm_valid_scores.npy"
)
ocsvm_test_scores = np.load(
    SCORES_PATH / "ocsvm_test_scores.npy"
)

ae_valid_scores = np.load(
    SCORES_PATH / "autoencoder_valid_scores.npy"
)
ae_test_scores = np.load(
    SCORES_PATH / "autoencoder_test_scores.npy"
)

print("Validation lengths:")
print(
    len(iforest_valid_scores),
    len(ocsvm_valid_scores),
    len(ae_valid_scores)
)

print("\nTest lengths:")
print(
    len(iforest_test_scores),
    len(ocsvm_test_scores),
    len(ae_test_scores)
)


Validation lengths:
108423 108423 108423

Test lengths:
111306 111306 111306


In [6]:
assert len(test_raw) == len(iforest_test_scores)
assert len(test_raw) == len(ocsvm_test_scores)
assert len(test_raw) == len(ae_test_scores)

assert len(valid_raw) == len(iforest_valid_scores)
assert len(valid_raw) == len(ocsvm_valid_scores)
assert len(valid_raw) == len(ae_valid_scores)

print("Raw rows and anomaly scores are aligned.")


Raw rows and anomaly scores are aligned.


## Load saved operational thresholds

In [7]:
with open(
    MODELS_PATH / "isolation_forest_thresholds.json",
    "r"
) as f:
    iforest_thresholds = json.load(f)

with open(
    MODELS_PATH / "one_class_svm_thresholds.json",
    "r"
) as f:
    ocsvm_thresholds = json.load(f)

with open(
    MODELS_PATH / "autoencoder_thresholds.json",
    "r"
) as f:
    ae_thresholds = json.load(f)

print("Isolation Forest:", iforest_thresholds)
print("\nOne-Class SVM:", ocsvm_thresholds)
print("\nAutoencoder:", ae_thresholds)


Isolation Forest: {'max_f1': {'threshold': -0.0701127042784469, 'validation_precision': 0.21054702872884692, 'validation_recall': 0.44583333333333336, 'validation_f1': 0.28601978080683504}, 'high_recall': {'target_recall': 0.8, 'threshold': -0.0830482062100314, 'validation_precision': 0.13677639046538026, 'validation_recall': 0.8033333333333333, 'validation_f1': 0.2337536372451442}}

One-Class SVM: {'max_f1': {'threshold': 23.177816949805475, 'validation_precision': 0.5544217687074829, 'validation_recall': 0.4075, 'validation_f1': 0.4697406340052753}, 'high_recall': {'target_recall': 0.8, 'threshold': -2.386012452009169, 'validation_precision': 0.10950153986540435, 'validation_recall': 0.8, 'validation_f1': 0.19263569780253728}}

Autoencoder: {'max_f1': {'threshold': 0.023564724251627922, 'validation_precision': 0.31672297297297297, 'validation_recall': 0.3125, 'validation_f1': 0.31459731543574165}, 'high_recall': {'target_recall': 0.8, 'threshold': 0.005704281851649284, 'validation_pr

## Convert anomaly scores to validation-referenced percentiles

The three models produce scores on very different numerical scales.

To make them comparable, each score is converted to a percentile relative to that model's **validation score distribution**.

Example:

- percentile `0.50` → around the middle of validation scores
- percentile `0.95` → more anomalous than about 95% of validation transactions
- percentile `0.99` → extremely unusual according to that model

No fraud labels are used for this normalization.


In [8]:
def percentile_from_reference(
    reference_scores,
    scores
):
    reference_sorted = np.sort(
        np.asarray(reference_scores)
    )

    ranks = np.searchsorted(
        reference_sorted,
        scores,
        side="right"
    )

    return ranks / len(reference_sorted)


In [9]:
iforest_test_pct = percentile_from_reference(
    iforest_valid_scores,
    iforest_test_scores
)

ocsvm_test_pct = percentile_from_reference(
    ocsvm_valid_scores,
    ocsvm_test_scores
)

ae_test_pct = percentile_from_reference(
    ae_valid_scores,
    ae_test_scores
)

pd.DataFrame({
    "Isolation Forest percentile": iforest_test_pct,
    "OCSVM percentile": ocsvm_test_pct,
    "Autoencoder percentile": ae_test_pct
}).describe()


,Isolation Forest percentile,OCSVM percentile,Autoencoder percentile
count,111306.000000,111306.000000,111306.000000
mean,0.487606,0.496039,0.489222
std,0.292321,0.288125,0.291681
min,0.002499,0.000009,0.000009
25%,0.232377,0.246696,0.233744
50%,0.476292,0.491063,0.480848
75%,0.742370,0.744911,0.739825
max,1.000000,1.000000,0.999991


## Build transaction-level risk table

In [10]:
risk_table = test_raw.reset_index().rename(
    columns={"index": "original_index"}
).copy()

risk_table["iforest_score"] = iforest_test_scores
risk_table["ocsvm_score"] = ocsvm_test_scores
risk_table["autoencoder_score"] = ae_test_scores

risk_table["iforest_percentile"] = iforest_test_pct
risk_table["ocsvm_percentile"] = ocsvm_test_pct
risk_table["autoencoder_percentile"] = ae_test_pct

risk_table.head()


,original_index,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud,iforest_score,ocsvm_score,autoencoder_score,iforest_percentile,ocsvm_percentile,autoencoder_percentile
0,483337,150,C748358246,2,M,28007,M1823072687,28007,es_transportation,23.74,0,-0.156539,-11.698156,0.000013,0.107523,0.288979,0.123885
1,483338,150,C1992960127,3,F,28007,M1823072687,28007,es_transportation,11.36,0,-0.155377,-16.859365,0.000023,0.191528,0.116322,0.675281
2,483339,150,C1435043122,5,M,28007,M1053599405,28007,es_health,227.76,0,-0.068409,4.433641,0.005250,0.979091,0.971454,0.919805
3,483340,150,C1435043122,5,M,28007,M1913465890,28007,es_health,150.86,0,-0.065376,-1.252596,0.011811,0.983979,0.937624,0.959879
4,483341,150,C2089485741,3,M,28007,M1823072687,28007,es_transportation,23.00,0,-0.153181,-10.740889,0.000023,0.285493,0.358512,0.693137


## Add model alerts

In [11]:
if_f1_threshold = iforest_thresholds[
    "max_f1"
]["threshold"]

if_high_threshold = iforest_thresholds[
    "high_recall"
]["threshold"]

ocsvm_f1_threshold = ocsvm_thresholds[
    "max_f1"
]["threshold"]

ocsvm_high_threshold = ocsvm_thresholds[
    "high_recall"
]["threshold"]

ae_f1_threshold = ae_thresholds[
    "max_f1"
]["threshold"]

ae_high_threshold = ae_thresholds[
    "high_recall"
]["threshold"]

risk_table["iforest_max_f1_alert"] = (
    risk_table["iforest_score"]
    >= if_f1_threshold
)

risk_table["iforest_high_recall_alert"] = (
    risk_table["iforest_score"]
    >= if_high_threshold
)

risk_table["ocsvm_max_f1_alert"] = (
    risk_table["ocsvm_score"]
    >= ocsvm_f1_threshold
)

risk_table["ocsvm_high_recall_alert"] = (
    risk_table["ocsvm_score"]
    >= ocsvm_high_threshold
)

risk_table["autoencoder_max_f1_alert"] = (
    risk_table["autoencoder_score"]
    >= ae_f1_threshold
)

risk_table["autoencoder_high_recall_alert"] = (
    risk_table["autoencoder_score"]
    >= ae_high_threshold
)


## Model agreement and risk tiers

In [12]:
# Number of models placing a transaction in the top 5%
# of their validation-referenced anomaly distribution.
risk_table["models_top_5pct"] = (
    (risk_table["iforest_percentile"] >= 0.95).astype(int)
    + (risk_table["ocsvm_percentile"] >= 0.95).astype(int)
    + (risk_table["autoencoder_percentile"] >= 0.95).astype(int)
)

# Simple consensus percentile, used only as investigation context.
risk_table["mean_anomaly_percentile"] = risk_table[
    [
        "iforest_percentile",
        "ocsvm_percentile",
        "autoencoder_percentile"
    ]
].mean(axis=1)


In [13]:
def assign_risk_tier(row):

    if (
        row["models_top_5pct"] == 3
        or row["ocsvm_max_f1_alert"]
    ):
        return "HIGH"

    if (
        row["models_top_5pct"] >= 2
        or row["iforest_high_recall_alert"]
        or row["autoencoder_high_recall_alert"]
    ):
        return "MEDIUM"

    return "LOW"


risk_table["risk_tier"] = risk_table.apply(
    assign_risk_tier,
    axis=1
)

risk_table["risk_tier"].value_counts()


risk_tier
LOW       100524
MEDIUM      8455
HIGH        2327
Name: count, dtype: int64

The risk tier is **not a newly trained fraud model**.

It is an investigation-routing rule that combines:
- model agreement
- the high-confidence One-Class SVM operating point
- the high-recall detector operating points

It is intended to prioritize analyst attention.


## Build training-only contextual reference statistics

The explanation layer should not use test-set information to decide what is unusual.

We therefore compute contextual baselines from the **training period only**.


In [14]:
train_reference = train_raw.copy()

train_reference["amount_log"] = np.log1p(
    train_reference["amount"]
)

global_amount_sorted = np.sort(
    train_reference["amount"].to_numpy()
)

category_counts = train_reference[
    "category"
].value_counts()

merchant_counts = train_reference[
    "merchant"
].value_counts()

n_train = len(train_reference)

category_frequency = (
    category_counts / n_train
)

merchant_frequency = (
    merchant_counts / n_train
)

print("Training transactions:", n_train)
print("Categories:", len(category_counts))
print("Merchants:", len(merchant_counts))


Training transactions: 374914
Categories: 15
Merchants: 50


## Precompute group-specific amount distributions

In [15]:
category_amount_reference = {
    category: np.sort(group["amount"].to_numpy())
    for category, group
    in train_reference.groupby("category")
}

merchant_amount_reference = {
    merchant: np.sort(group["amount"].to_numpy())
    for merchant, group
    in train_reference.groupby("merchant")
}


## Helper: empirical percentile

In [16]:
def empirical_percentile(
    sorted_reference,
    value
):
    if len(sorted_reference) == 0:
        return np.nan

    rank = np.searchsorted(
        sorted_reference,
        value,
        side="right"
    )

    return rank / len(sorted_reference)


## Explain one transaction using historical context

In [17]:
def contextual_explanation(row):

    amount = float(row["amount"])
    category = row["category"]
    merchant = row["merchant"]

    global_amount_pct = empirical_percentile(
        global_amount_sorted,
        amount
    )

    category_amount_pct = empirical_percentile(
        category_amount_reference.get(
            category,
            np.array([])
        ),
        amount
    )

    merchant_amount_pct = empirical_percentile(
        merchant_amount_reference.get(
            merchant,
            np.array([])
        ),
        amount
    )

    cat_freq = float(
        category_frequency.get(
            category,
            0.0
        )
    )

    merchant_freq = float(
        merchant_frequency.get(
            merchant,
            0.0
        )
    )

    reasons = []

    if global_amount_pct >= 0.99:
        reasons.append(
            "Amount is above the 99th percentile "
            "of training transactions."
        )
    elif global_amount_pct >= 0.95:
        reasons.append(
            "Amount is above the 95th percentile "
            "of training transactions."
        )

    if category_amount_pct >= 0.99:
        reasons.append(
            "Amount is unusually high for this category."
        )
    elif category_amount_pct >= 0.95:
        reasons.append(
            "Amount is high relative to transactions "
            "in the same category."
        )

    if merchant_amount_pct >= 0.99:
        reasons.append(
            "Amount is unusually high for this merchant."
        )
    elif merchant_amount_pct >= 0.95:
        reasons.append(
            "Amount is high relative to transactions "
            "for the same merchant."
        )

    if cat_freq <= 0.005:
        reasons.append(
            "Transaction category is rare in training history."
        )

    if merchant_freq <= 0.005:
        reasons.append(
            "Merchant is rare in training history."
        )

    if row["models_top_5pct"] == 3:
        reasons.append(
            "All three anomaly detectors place the "
            "transaction in their top 5% most anomalous region."
        )
    elif row["models_top_5pct"] == 2:
        reasons.append(
            "Two anomaly detectors place the transaction "
            "in their top 5% most anomalous region."
        )

    if row["ocsvm_max_f1_alert"]:
        reasons.append(
            "One-Class SVM triggered its high-confidence alert threshold."
        )

    if not reasons:
        reasons.append(
            "No single contextual rule dominates; "
            "the anomaly is primarily model-driven."
        )

    return {
        "global_amount_percentile":
            float(global_amount_pct),

        "category_amount_percentile":
            float(category_amount_pct)
            if not np.isnan(category_amount_pct)
            else None,

        "merchant_amount_percentile":
            float(merchant_amount_pct)
            if not np.isnan(merchant_amount_pct)
            else None,

        "category_frequency":
            cat_freq,

        "merchant_frequency":
            merchant_freq,

        "reasons":
            reasons
    }


## Inspect the highest-priority transactions

In [18]:
priority_transactions = risk_table.sort_values(
    by=[
        "risk_tier",
        "models_top_5pct",
        "mean_anomaly_percentile"
    ],
    ascending=[
        True,
        False,
        False
    ]
).copy()

# Explicit risk ordering
risk_order = {
    "HIGH": 0,
    "MEDIUM": 1,
    "LOW": 2
}

priority_transactions["_risk_order"] = (
    priority_transactions["risk_tier"]
    .map(risk_order)
)

priority_transactions = priority_transactions.sort_values(
    by=[
        "_risk_order",
        "models_top_5pct",
        "mean_anomaly_percentile"
    ],
    ascending=[
        True,
        False,
        False
    ]
).drop(
    columns="_risk_order"
)

priority_transactions[
    [
        "step",
        "customer",
        "merchant",
        "category",
        "amount",
        "risk_tier",
        "models_top_5pct",
        "mean_anomaly_percentile"
    ]
].head(20)


,step,customer,merchant,category,amount,risk_tier,models_top_5pct,mean_anomaly_percentile
78410,171,C1837321151,M732195782,es_travel,1796.07,HIGH,3,0.999720
66192,167,C1871125244,M1888755466,es_otherservices,294.20,HIGH,3,0.999437
21450,155,C808326652,M85975013,es_food,0.79,HIGH,3,0.999394
68186,168,C2094361013,M1053599405,es_health,2.37,HIGH,3,0.999308
7851,152,C808326652,M1913465890,es_health,204.52,HIGH,3,0.998902
24227,156,C808326652,M1313686961,es_contents,17.76,HIGH,3,0.998859
85335,173,C762614808,M732195782,es_travel,5772.54,HIGH,3,0.998826
40601,160,C1374607221,M677738360,es_contents,30.95,HIGH,3,0.998795
29026,157,C914000857,M840466850,es_tech,151.13,HIGH,3,0.998776
57115,165,C2075935351,M1600850729,es_fashion,106.47,HIGH,3,0.998669


## Example contextual explanation

In [19]:
example_row = priority_transactions.iloc[0]

example_context = contextual_explanation(
    example_row
)

print("Transaction:")
print(
    example_row[
        [
            "step",
            "customer",
            "merchant",
            "category",
            "amount",
            "risk_tier"
        ]
    ]
)

print("\nContext:")
print(
    json.dumps(
        example_context,
        indent=2
    )
)


Transaction:
step                 171
customer     C1837321151
merchant      M732195782
category       es_travel
amount           1796.07
risk_tier           HIGH
Name: 78410, dtype: object

Context:
{
  "global_amount_percentile": 0.9993225112959239,
  "category_amount_percentile": 0.48268839103869654,
  "merchant_amount_percentile": 0.45363408521303256,
  "category_frequency": 0.0013096336759896936,
  "merchant_frequency": 0.0010642440666392826,
  "reasons": [
    "Amount is above the 99th percentile of training transactions.",
    "Transaction category is rare in training history.",
    "Merchant is rare in training history.",
    "All three anomaly detectors place the transaction in their top 5% most anomalous region.",
    "One-Class SVM triggered its high-confidence alert threshold."
  ]
}


## Autoencoder feature-level reconstruction explanation

The Autoencoder gives us a second type of explanation.

For each transformed feature, we can compute:

`(input_feature - reconstructed_feature)^2`

The features with the largest reconstruction errors are the dimensions the network had the most difficulty reconstructing.


In [20]:
with open(
    MODELS_PATH / "feature_names.json",
    "r"
) as f:
    full_feature_names = json.load(f)

step_index = full_feature_names.index(
    "num__step"
)

ae_keep_indices = [
    i for i in range(len(full_feature_names))
    if i != step_index
]

ae_feature_names = [
    full_feature_names[i]
    for i in ae_keep_indices
]

X_test = sparse.load_npz(
    PROCESSED_PATH / "X_test.npz"
)

X_test_ae = X_test[
    :,
    ae_keep_indices
]


In [21]:
class Autoencoder(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 12),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(12, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent)


In [22]:
device = torch.device("cpu")

autoencoder = Autoencoder(
    input_dim=len(ae_feature_names)
).to(device)

state_dict = torch.load(
    MODELS_PATH / "autoencoder_state_dict.pt",
    map_location=device
)

autoencoder.load_state_dict(
    state_dict
)

autoencoder.eval()

print("Autoencoder loaded.")


Autoencoder loaded.


## Feature reconstruction errors for one transaction

In [23]:
def autoencoder_feature_explanation(
    test_position,
    top_n=10
):
    x = X_test_ae[
        test_position
    ].toarray().astype(np.float32)

    x_tensor = torch.from_numpy(
        x
    ).to(device)

    with torch.no_grad():
        reconstructed = autoencoder(
            x_tensor
        ).cpu().numpy()[0]

    original = x[0]

    squared_error = (
        original - reconstructed
    ) ** 2

    explanation = pd.DataFrame({
        "feature": ae_feature_names,
        "input_value": original,
        "reconstructed_value": reconstructed,
        "squared_error": squared_error
    })

    return explanation.sort_values(
        "squared_error",
        ascending=False
    ).head(top_n)


In [24]:
example_position = int(
    priority_transactions.index[0]
)

ae_feature_explanation = (
    autoencoder_feature_explanation(
        example_position,
        top_n=10
    )
)

ae_feature_explanation


,feature,input_value,reconstructed_value,squared_error
1,cat__age_0,1.0,0.058135,0.887110
53,cat__merchant_M732195782,1.0,0.064546,0.875075
76,cat__category_es_travel,1.0,0.069803,0.865266
74,cat__category_es_tech,0.0,0.214924,0.046192
7,cat__age_6,0.0,0.186685,0.034851
68,cat__category_es_home,0.0,0.150001,0.022500
5,cat__age_4,0.0,0.143986,0.020732
6,cat__age_5,0.0,0.143823,0.020685
69,cat__category_es_hotelservices,0.0,0.136371,0.018597
4,cat__age_3,0.0,0.128534,0.016521


Feature names such as:

- `num__amount_log`
- `cat__category_...`
- `cat__merchant_...`

show which transformed dimensions contributed most to the reconstruction anomaly.

This is not a causal explanation. It is a **local reconstruction-based explanation**.


## Build structured investigation packets

In [25]:
def build_investigation_packet(
    test_position,
    include_label=False,
    top_ae_features=5
):
    row = risk_table.iloc[
        test_position
    ]

    context = contextual_explanation(
        row
    )

    ae_features = (
        autoencoder_feature_explanation(
            test_position,
            top_n=top_ae_features
        )
    )

    transaction = {
        "step": int(row["step"]),
        "customer": str(row["customer"]),
        "merchant": str(row["merchant"]),
        "category": str(row["category"]),
        "amount": float(row["amount"]),
        "age": str(row["age"]),
        "gender": str(row["gender"])
    }

    model_scores = {
        "isolation_forest": {
            "score": float(
                row["iforest_score"]
            ),
            "percentile": float(
                row["iforest_percentile"]
            ),
            "max_f1_alert": bool(
                row["iforest_max_f1_alert"]
            ),
            "high_recall_alert": bool(
                row["iforest_high_recall_alert"]
            )
        },

        "one_class_svm": {
            "score": float(
                row["ocsvm_score"]
            ),
            "percentile": float(
                row["ocsvm_percentile"]
            ),
            "max_f1_alert": bool(
                row["ocsvm_max_f1_alert"]
            ),
            "high_recall_alert": bool(
                row["ocsvm_high_recall_alert"]
            )
        },

        "autoencoder": {
            "score": float(
                row["autoencoder_score"]
            ),
            "percentile": float(
                row["autoencoder_percentile"]
            ),
            "max_f1_alert": bool(
                row["autoencoder_max_f1_alert"]
            ),
            "high_recall_alert": bool(
                row["autoencoder_high_recall_alert"]
            )
        }
    }

    packet = {
        "transaction_id": int(
            row["original_index"]
        ),
        "risk_tier": row["risk_tier"],
        "models_top_5pct": int(
            row["models_top_5pct"]
        ),
        "mean_anomaly_percentile": float(
            row["mean_anomaly_percentile"]
        ),
        "transaction": transaction,
        "model_scores": model_scores,
        "context": context,
        "autoencoder_top_reconstruction_features":
            ae_features.to_dict(
                orient="records"
            )
    }

    if include_label:
        packet["ground_truth_fraud"] = int(
            row["fraud"]
        )

    return packet


By default, the packet used for an LLM should **not include the ground-truth fraud label**.

The label can be added only when evaluating whether explanations correspond to known fraud cases.


In [26]:
example_packet = build_investigation_packet(
    test_position=example_position,
    include_label=False,
    top_ae_features=5
)

print(
    json.dumps(
        example_packet,
        indent=2
    )
)


{
  "transaction_id": 561747,
  "risk_tier": "HIGH",
  "models_top_5pct": 3,
  "mean_anomaly_percentile": 0.9997202315621839,
  "transaction": {
    "step": 171,
    "customer": "C1837321151",
    "merchant": "M732195782",
    "category": "es_travel",
    "amount": 1796.07,
    "age": "0",
    "gender": "M"
  },
  "model_scores": {
    "isolation_forest": {
      "score": -0.021856840952361234,
      "percentile": 0.9998708761056233,
      "max_f1_alert": true,
      "high_recall_alert": true
    },
    "one_class_svm": {
      "score": 124.66078661261568,
      "percentile": 0.9994835044224933,
      "max_f1_alert": true,
      "high_recall_alert": true
    },
    "autoencoder": {
      "score": 0.037667468190193176,
      "percentile": 0.999806314158435,
      "max_f1_alert": true,
      "high_recall_alert": true
    }
  },
  "context": {
    "global_amount_percentile": 0.9993225112959239,
    "category_amount_percentile": 0.48268839103869654,
    "merchant_amount_percentile": 0.4536

## Export top investigation cases

In [27]:
TOP_N = 100

top_positions = (
    priority_transactions
    .head(TOP_N)
    .index
    .tolist()
)

investigation_packets = [
    build_investigation_packet(
        test_position=int(position),
        include_label=False,
        top_ae_features=5
    )
    for position in top_positions
]

output_json = (
    INVESTIGATION_PATH
    / "top_100_investigation_packets.json"
)

with open(
    output_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        investigation_packets,
        f,
        indent=2
    )

print("Saved:", output_json)


Saved: ..\results\investigations\top_100_investigation_packets.json


## Export compact risk table

In [28]:
risk_export_columns = [
    "original_index",
    "step",
    "customer",
    "merchant",
    "category",
    "amount",
    "risk_tier",
    "models_top_5pct",
    "mean_anomaly_percentile",
    "iforest_percentile",
    "ocsvm_percentile",
    "autoencoder_percentile",
    "iforest_max_f1_alert",
    "ocsvm_max_f1_alert",
    "autoencoder_max_f1_alert",
    "iforest_high_recall_alert",
    "ocsvm_high_recall_alert",
    "autoencoder_high_recall_alert"
]

risk_export = risk_table[
    risk_export_columns
].copy()

risk_output = (
    INVESTIGATION_PATH
    / "test_transaction_risk_table.csv"
)

risk_export.to_csv(
    risk_output,
    index=False
)

print("Saved:", risk_output)


Saved: ..\results\investigations\test_transaction_risk_table.csv


## Sanity check against ground truth

The `fraud` label is used here **only for evaluation**, not for creating the explanation.


In [29]:
evaluation_by_tier = (
    risk_table
    .groupby("risk_tier")["fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
    .sort_index()
)

evaluation_by_tier


,transactions,frauds,fraud_rate
risk_tier,,,
HIGH,2327,705,0.302965
LOW,100524,62,0.000617
MEDIUM,8455,433,0.051212


In [30]:
agreement_evaluation = (
    risk_table
    .groupby("models_top_5pct")["fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
)

agreement_evaluation


,transactions,frauds,fraud_rate
models_top_5pct,,,
0,101693,46,0.000452
1,4565,208,0.045564
2,3021,363,0.120159
3,2027,583,0.287617


In [31]:
medium_index = risk_table.index[
    risk_table["risk_tier"] == "MEDIUM"
][0]

high_index = risk_table.index[
    risk_table["risk_tier"] == "HIGH"
][0]

print("MEDIUM test index:", medium_index)
print("HIGH test index  :", high_index)

MEDIUM test index: 2
HIGH test index  : 69


## Conclusion

This notebook creates a bridge between anomaly detection and investigation.

For every suspicious transaction we can now provide:

1. **raw transaction context**
2. **three independent anomaly scores**
3. **validation-referenced anomaly percentiles**
4. **model agreement**
5. **operational alert status**
6. **amount rarity relative to historical training data**
7. **category and merchant rarity**
8. **Autoencoder feature-level reconstruction errors**
9. **a structured JSON investigation packet**

The next step is to use these packets as controlled input to an LLM that generates a concise fraud-investigation report without allowing the LLM to redefine the anomaly score or ground-truth label.
